[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/08_Your_First_Knowledge_Graph.ipynb)

# 🚀 你的第一个知识图谱

## 概述

本 Notebook 带你从一份简单的文档开始创建你的第一个知识图谱。你将学习从接入文件到可视化最终知识图谱的完整端到端工作流 —— 并且每一步都使用上一步的真实输出。

> [!TIP]
> 如果你刚接触 Semantica,这是完美的起点。无需任何知识图谱的预备知识!

**文档**:[API 参考](https://semantica.readthedocs.io/reference/kg/)

### 🎯 学习目标

- **理解工作流**:学习 `文件 → 解析 → 抽取 → 图 → 可视化` 的管线
- **接入数据**:使用 `FileIngestor` 加载文档
- **解析内容**:使用 `DocumentParser` 提取文本
- **抽取知识**:使用 `NERExtractor` 和 `RelationExtractor` 识别实体和关系
- **构建图**:使用 `GraphBuilder` 构建图
- **可视化**:使用 `KGVisualizer` 让你的图生动起来

## 安装

从 PyPI 安装 Semantica:

```bash
pip install semantica
# 或安装全部可选依赖:
pip install semantica[all]
```

---

## 🔄 简单的端到端工作流

完整的工作流由五个主要步骤组成:

1. **📥 接入** - 从文件或其他来源加载数据
2. **📄 解析** - 从文档中提取并结构化内容
3. **⛏️ 抽取** - 识别实体和关系
4. **🕸️ 构建图** - 构建知识图谱
5. **📊 可视化** - 渲染并分析图

下面的代码单元格逐一演示了每个步骤,并且每个单元格都可以单独重新运行:示例文件只会被最后的可选清理单元格删除。

> [!TIP]
> **替代方案:使用 Semantica 框架**
>
> 若想采用更简单的高层方式,你可以使用 `Semantica` 框架类,它会编排所有这些步骤:
>
> ```python
> from semantica.core import Semantica
>
> framework = Semantica()
> framework.initialize()
>
> result = framework.build_knowledge_base(
>     sources=["sample_document.txt"],
>     embeddings=True,
>     graph=True
> )
>
> framework.shutdown()
> ```
>
> 本 Notebook 展示的是便于学习的分步方式。更多细节见[核心模块使用指南](../../../semantica/core/core_usage.md)。

---

## 📂 步骤 1:接入文件

在这一步中,我们使用 `FileIngestor` 加载一份文档。接入器支持多种文件格式,包括 PDF、DOCX、TXT 等。写入示例文件的操作是幂等的,因此该单元格可随时重新运行。

In [ ]:
%pip install semantica

# spaCy models are distributed separately from the spaCy library. This lesson
# relies on the English model to recognize standalone places such as Cupertino.
import sys
import subprocess
import spacy

try:
    spacy.load("en_core_web_sm")
except OSError:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])


In [ ]:
from pathlib import Path

from semantica.ingest import FileIngestor

sample_text = """Apple Inc. is headquartered in Cupertino, California.
In 1976, Steve Jobs founded Apple Inc.
Tim Cook is the CEO of Apple Inc.
"""

sample_file = Path("sample_document.txt")
sample_file.write_text(sample_text)

print(f"File: {sample_file}")
print(f"Content length: {len(sample_text)} characters")

ingestor = FileIngestor()
file_object = ingestor.ingest_file(sample_file, read_content=True)
print(f"  File name: {file_object.name}")
print(f"  File type: {file_object.file_type}")
print(f"  Content available: {file_object.content is not None}")

## 📄 步骤 2:解析文档

接入文件之后,我们需要解析它以提取文本内容。`DocumentParser.parse_document()` 会在 `"text"` 键下返回提取出的文本。

In [ ]:
from semantica.parse import DocumentParser

parser = DocumentParser()
parsed_document = parser.parse_document(str(sample_file))

parsed_content = parsed_document.get("text", "")
assert parsed_content.strip(), "Parsing produced no text — check the input file"

print(f"Parsed content length: {len(parsed_content)} characters")
print(f"Preview: {parsed_content[:120]}...")

## ⛏️ 步骤 3:抽取实体与关系

现在我们从解析后的文本中抽取实体和关系。`NERExtractor` 识别人物、组织、地点和日期;`RelationExtractor` 发现这些提及之间的关系。两者都作用于*步骤 2 中解析出的内容* —— 而不是原始字符串的副本。

In [ ]:
from semantica.semantic_extract import NERExtractor, RelationExtractor

ner_extractor = NERExtractor()
relation_extractor = RelationExtractor()

mentions = ner_extractor.extract(parsed_content)
relations = relation_extractor.extract(parsed_content, mentions)

print("Entity mentions:")
for mention in mentions:
    print(f"  {mention.text!r:<13} {mention.label:<7} span=[{mention.start_char}:{mention.end_char}]")

print("\nExtracted relations:")
for rel in relations:
    print(f"  {rel.subject.text!r} --{rel.predicate}--> {rel.object.text!r}")

## 🕸️ 步骤 4:构建知识图谱

利用抽取出的实体和关系,我们使用 `GraphBuilder` 构建知识图谱。每个提及都会获得一个图 ID,并且每条边都是根据实际的 `Relation.subject` / `Relation.object` 端点构建的。

> [!NOTE]
> 图中每个*提及*对应一个节点,因此 `Apple Inc.` 会出现三次。把重复的提及合并为一个规范实体,会在 [07_Building_Knowledge_Graphs.ipynb](./07_Building_Knowledge_Graphs.ipynb) 中介绍。

In [ ]:
from semantica.kg import GraphBuilder

entities = []
span_to_id = {}
for i, mention in enumerate(mentions, 1):
    graph_id = f"e{i}"
    span_to_id[(mention.start_char, mention.end_char)] = graph_id
    entities.append({
        "id": graph_id,
        "type": mention.label,
        "name": mention.text,
        "properties": {},
    })

relationships = []
for rel in relations:
    source_id = span_to_id.get((rel.subject.start_char, rel.subject.end_char))
    target_id = span_to_id.get((rel.object.start_char, rel.object.end_char))
    if source_id is None or target_id is None:
        print(f"Skipping relation with unmapped endpoint: "
              f"{rel.subject.text!r} --{rel.predicate}--> {rel.object.text!r}")
        continue
    relationships.append({
        "source": source_id,
        "target": target_id,
        "type": rel.predicate,
        "properties": {},
    })

builder = GraphBuilder()
knowledge_graph = builder.build({"entities": entities, "relationships": relationships})

id_to_name = {entity["id"]: entity["name"] for entity in entities}

print(f"Nodes (entities): {len(knowledge_graph['entities'])}")
for entity in knowledge_graph["entities"]:
    print(f"  {entity['id']}: {entity['name']} ({entity['type']})")

print(f"\nEdges (relationships): {len(knowledge_graph['relationships'])}")
for relationship in knowledge_graph["relationships"]:
    print(f"  {id_to_name[relationship['source']]} "
          f"--{relationship['type']}--> {id_to_name[relationship['target']]}")

edges = {
    (id_to_name[r["source"]], r["type"], id_to_name[r["target"]])
    for r in knowledge_graph["relationships"]
}
assert ("Apple Inc.", "located_in", "Cupertino") in edges
assert ("Tim Cook", "works_for", "Apple Inc.") in edges

## 📊 步骤 5:可视化与分析

最后,我们使用 `KGVisualizer` 渲染知识图谱并查看其结构。`visualize_network()` 可直接接受 `GraphBuilder` 的结果,并能保存为交互式 HTML 文件。

In [ ]:
from semantica.visualization import KGVisualizer

visualizer = KGVisualizer()
fig = visualizer.visualize_network(
    knowledge_graph, output="html", file_path="knowledge_graph.html"
)
print("Saved interactive visualization to knowledge_graph.html")

entity_types = {}
for entity in knowledge_graph["entities"]:
    entity_types[entity["type"]] = entity_types.get(entity["type"], 0) + 1

print("\nEntities by type:")
for entity_type, count in sorted(entity_types.items()):
    print(f"  - {entity_type}: {count}")

relationship_types = {}
for relationship in knowledge_graph["relationships"]:
    relationship_types[relationship["type"]] = (
        relationship_types.get(relationship["type"], 0) + 1
    )

print("\nRelationships by type:")
for relationship_type, count in sorted(relationship_types.items()):
    print(f"  - {relationship_type}: {count}")

fig

## 🧹 可选:清理

只有在你用完本 Notebook 之后才运行这个单元格。前面的单元格会读取 `sample_document.txt`,因此在你于此处将其删除之前,它们都可以重复运行。

In [ ]:
for path in [sample_file, Path("knowledge_graph.html")]:
    if path.exists():
        path.unlink()
        print(f"Removed {path}")

## 总结

你已经端到端地构建了自己的第一个知识图谱:

- **FileIngestor** 加载了示例文档
- **DocumentParser** 在 `"text"` 键下返回了文档文本
- **NERExtractor** / **RelationExtractor** 从该文本中产出了真实的提及和关系
- **GraphBuilder** 把它们转换为一张图,其边来自实际的关系端点
- **KGVisualizer** 将结果渲染为交互式网络

下一步:在 [07_Building_Knowledge_Graphs.ipynb](./07_Building_Knowledge_Graphs.ipynb) 中使用 `EntityResolver` 合并重复的提及,或在 Graph Analytics Notebook 中探索图指标。